# Winery E-commerce — Data Cleaning & Modeling

Starting from an anonymized dataset (no customer PII, no real product names/SKUs).

In [0]:
import pandas as pd

df_valid = pd.read_csv(
    '../data/processed/df_valid_anonymized.csv',
    low_memory=False
)

print(df_valid.shape)
df_valid.head()

In [0]:
df_valid.dtypes

## Date validation

In [0]:
# define which columns should be parsed back into real dates
date_cols = ['order_date', 'paid_date']

# parse each date column, turning any invalid/unparseable value into NaT instead of raising an error
for col_name in date_cols:
    df_valid[col_name] = pd.to_datetime(df_valid[col_name], errors='coerce')
    n_invalid = df_valid[col_name].isna().sum()
    print(f"{col_name}: {n_invalid} invalid/missing dates")

In [0]:
# check which order statuses correspond to the missing paid_date values
df_valid[df_valid['paid_date'].isna()]['status'].value_counts()

## Order status filtering

In [0]:
# check the full status distribution across all orders
df_valid["status"].value_counts()

In [0]:
n_before = len(df_valid)

# keep only orders with status "completed", since only these represent real, finalized revenue
df_valid = df_valid[df_valid["status"] == "completed"].copy()

n_excluded = n_before - len(df_valid)
print(f"{n_excluded} orders excluded (status != completed)")

## Structure & overview

In [0]:
# check the number of rows and columns in the cleaned dataset
df_valid.shape

In [0]:
# check column names and their inferred data types
df_valid.dtypes

In [0]:
# look at a sample of rows
df_valid.head(10)

In [0]:
# a random sample avoids being misled by the first rows only, which can be unrepresentative
df_valid.sample(5)

In [0]:
# get a concise summary: column names, non-null counts, and dtypes in one view
df_valid.info()

### Redundant columns

In [0]:
# check whether order_id and order_number are always identical
(df_valid['order_id'] == df_valid['order_number']).all()

In [0]:
# I drop one of the redundant columns
df_valid = df_valid.drop(columns=["order_number"])

In [0]:
# check if order_currency has more than one distinct value
df_valid['order_currency'].value_counts()

In [0]:
# FIX: shipping_postcode came back as float64 after the CSV round-trip, which loses
# leading zeros in Italian postcodes (e.g. "00100" -> 100.0). Casting back to string.
df_valid['shipping_postcode'] = df_valid['shipping_postcode'].astype('Int64').astype(str)

df_valid['shipping_postcode'].head()

In [0]:
# FIX: drop list reduced to only the columns that still exist at this stage
# (everything else — customer_user_agent, item_meta, stripe/paypal fees, etc. —
# was already removed in the private de-identification step, before this CSV was created)
df_valid = df_valid.drop(columns=[
    "tax_items",
    "fee_items",
    "coupon_items",
    "refund_items",
    "item_refunded",
    "item_refunded_qty",
    "transaction_id",
    "order_currency"
])

In [0]:
# convert order_id from float to integer first (removes the trailing .0), then to string
df_valid['order_id'] = df_valid['order_id'].astype('Int64').astype(str)

In [0]:
df_valid.info()